In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!pip -q install requests tqdm

In [ ]:
from google.colab import userdata
import os

# Your secret name is Canvas_API
os.environ["CANVAS_TOKEN"] = userdata.get("Canvas_API")

# Correct Canvas base URL from your screenshot
os.environ["CANVAS_BASE_URL"] = "https://kenan-flagler.instructure.com"

print("Loaded token:", "Yes" if os.environ["CANVAS_TOKEN"] else "No")
print("Base URL:", os.environ["CANVAS_BASE_URL"])

In [ ]:
# ===============================
# Canvas Authentication Loader
# ===============================

from google.colab import userdata
import os
import getpass

def load_canvas_auth(
    base_url: str = None,
    secret_name: str = "Canvas_API",
    allow_prompt: bool = True
):
    """
    Loads Canvas authentication safely.

    Priority:
    1) Colab Secret (secret_name)
    2) Environment variable CANVAS_TOKEN
    3) Secure prompt input (if allow_prompt=True)

    Sets:
        CANVAS_BASE_URL
        CANVAS_TOKEN
    """

    # Default UNC base URL if none provided
    if not base_url:
        base_url = "https://kenan-flagler.instructure.com"

    os.environ["CANVAS_BASE_URL"] = base_url.rstrip("/")

    token = None

    # 1️⃣ Try Colab Secrets
    try:
        token = userdata.get(secret_name)
    except Exception:
        token = None

    # 2️⃣ Try environment variable
    if not token:
        token = os.environ.get("CANVAS_TOKEN")

    # 3️⃣ Secure prompt fallback
    if (not token) and allow_prompt:
        token = getpass.getpass("Paste your Canvas API token (input hidden): ").strip()

    if not token:
        raise ValueError(
            "❌ No Canvas token found.\n"
            "Add it to Colab Secrets OR paste it when prompted."
        )

    os.environ["CANVAS_TOKEN"] = token.strip()

    print("✅ Canvas token loaded")
    print("🌐 Base URL:", os.environ["CANVAS_BASE_URL"])


# ===============================
# Run authentication
# ===============================

load_canvas_auth(secret_name="Canvas_API")

In [ ]:
import requests, os

base = os.environ["CANVAS_BASE_URL"]
token = os.environ["CANVAS_TOKEN"]

session = requests.Session()
session.headers.update({"Authorization": f"Bearer {token}"})

def get_all(url, params=None):
    params = params or {}
    params.setdefault("per_page", 100)
    results = []
    while url:
        r = session.get(url, params=params)
        r.raise_for_status()
        results.extend(r.json())
        link = r.headers.get("Link", "")
        next_url = None
        for part in link.split(","):
            if 'rel="next"' in part:
                next_url = part[part.find("<")+1:part.find(">")]
                break
        url = next_url
        params = None
    return results

courses = get_all(f"{base}/api/v1/courses", params={"include[]": "term"})
print("Total courses:", len(courses))
for c in courses:
    print(f"{c['id']} | {c.get('name')}")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
def categorize_file(filename: str) -> str:
    n = (filename or "").lower()

    # ---- assignments ----
    if ("assignment" in n) or n.startswith("hw") or n.startswith("homework") \
       or ("problem set" in n) or ("ps" in n):
        return "assignments"

    # ---- solutions ----
    if ("solution" in n) or ("solutions" in n) or ("answer" in n) or ("key" in n):
        return "solutions"

    # ---- readings / cases ----
    if ("case" in n) or ("reading" in n) or ("article" in n) \
       or ("paper" in n) or ("chapter" in n):
        return "readings"

    # ---- datasets ----
    if n.endswith((".csv", ".xlsx", ".xls", ".sav", ".dta")) \
       or ("data" in n) or ("dataset" in n):
        return "datasets"

    # ---- slides explicitly detected ----
    if n.endswith((".ppt", ".pptx", ".key")) \
       or ("slide" in n) or ("deck" in n) or ("lecture" in n):
        return "class_slides"

    # ⭐ DEFAULT FALLBACK
    return "class_slides"

The one below is for test on 1 course - in this course its operations

In [ ]:
import os, re, json
from pathlib import Path
import requests
from tqdm import tqdm

base = os.environ["CANVAS_BASE_URL"].rstrip("/")
token = os.environ["CANVAS_TOKEN"].strip()

COURSE_ID = 4038691  # <-- CHANGE THIS to your chosen course id

OUT_ROOT = Path("/content/drive/MyDrive/UNC_Archive_TEST_ONE_COURSE")

session = requests.Session()
session.headers.update({"Authorization": f"Bearer {token}"})

def safe_name(x: str) -> str:
    x = re.sub(r"[^\w\-. ]+", "_", (x or "").strip())
    return (x[:140] if len(x) > 140 else x) or "untitled"

def get_all(url, params=None):
    params = params or {}
    params.setdefault("per_page", 100)
    out = []
    while url:
        r = session.get(url, params=params)
        r.raise_for_status()
        out.extend(r.json())
        link = r.headers.get("Link", "")
        nxt = None
        for part in link.split(","):
            if 'rel="next"' in part:
                nxt = part[part.find("<")+1:part.find(">")]
                break
        url = nxt
        params = None
    return out

def download(url: str, dest: Path):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0:
        return "skipped"
    with session.get(url, stream=True) as r:
        r.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
    return "downloaded"

failed = []
downloaded_count = 0
skipped_count = 0

# Course details
course = session.get(f"{base}/api/v1/courses/{COURSE_ID}").json()
course_name = safe_name(course.get("name") or f"course_{COURSE_ID}")
course_dir = OUT_ROOT / course_name
course_dir.mkdir(parents=True, exist_ok=True)

(course_dir / "course.json").write_text(json.dumps(course, indent=2), encoding="utf-8")
print("Archiving:", course_name)
print("Output:", course_dir)

# 1) Files
files = get_all(f"{base}/api/v1/courses/{COURSE_ID}/files")
(course_dir / "files.json").write_text(json.dumps(files, indent=2), encoding="utf-8")

files_dir = course_dir / "files"
files_dir.mkdir(parents=True, exist_ok=True)

for fobj in tqdm(files, desc="Downloading course files", unit="file"):
    name = safe_name(fobj.get("display_name") or fobj.get("filename") or f"file_{fobj['id']}")
    url = fobj.get("url")
    if not url:
        continue

    try:
        category = categorize_file(name)
        dest = files_dir / category / name

        status = download(url, dest)

        if status == "downloaded":
            downloaded_count += 1
        else:
            skipped_count += 1

    except Exception as e:
        failed.append({"type": "file", "name": name, "error": str(e)})

# 2) Syllabus HTML
syllabus_html = course.get("syllabus_body") or ""
(course_dir / "syllabus.html").write_text(syllabus_html, encoding="utf-8")

# 3) Modules + external links
try:
    modules = get_all(f"{base}/api/v1/courses/{COURSE_ID}/modules")
    mod_dir = course_dir / "modules"
    mod_dir.mkdir(parents=True, exist_ok=True)
    external = set()

    for m in modules:
        mid = m["id"]
        items = get_all(f"{base}/api/v1/courses/{COURSE_ID}/modules/{mid}/items")
        (mod_dir / f"module_{mid}.json").write_text(
            json.dumps({"module": m, "items": items}, indent=2), encoding="utf-8"
        )
        for it in items:
            if it.get("external_url"):
                external.add(it["external_url"])

    (course_dir / "external_links.json").write_text(
        json.dumps(sorted(external), indent=2), encoding="utf-8"
    )
except Exception as e:
    failed.append({"type":"modules","error":str(e)})

# 4) Pages (optional)
try:
    pages = get_all(f"{base}/api/v1/courses/{COURSE_ID}/pages")
    pages_dir = course_dir / "pages"
    pages_dir.mkdir(parents=True, exist_ok=True)

    for p in pages:
        page = session.get(f"{base}/api/v1/courses/{COURSE_ID}/pages/{p['url']}").json()
        title = safe_name(page.get("title") or p["url"])
        (pages_dir / f"{title}.html").write_text(page.get("body") or "", encoding="utf-8")
except Exception as e:
    failed.append({"type":"pages","error":str(e)})

# 5) Assignments + submissions (best-effort)
try:
    assignments = get_all(f"{base}/api/v1/courses/{COURSE_ID}/assignments")
    (course_dir / "assignments.json").write_text(json.dumps(assignments, indent=2), encoding="utf-8")

    subs_dir = course_dir / "submissions"
    subs_dir.mkdir(parents=True, exist_ok=True)

    for a in tqdm(assignments, desc="Submissions (best-effort)", unit="asg"):
        aid = a["id"]
        try:
            subs = get_all(
                f"{base}/api/v1/courses/{COURSE_ID}/assignments/{aid}/submissions",
                params={"include[]": ["attachments", "submission_history"]}
            )
            (subs_dir / f"assignment_{aid}_submissions.json").write_text(
                json.dumps(subs, indent=2), encoding="utf-8"
            )

            for sub in subs:
                for att in (sub.get("attachments") or []):
                    att_url = att.get("url")
                    att_name = safe_name(att.get("filename") or f"attachment_{att.get('id')}")
                    if att_url:
                        status = download(att_url, subs_dir / str(aid) / att_name)
                        if status == "downloaded":
                            downloaded_count += 1
                        else:
                            skipped_count += 1
        except Exception:
            continue
except Exception as e:
    failed.append({"type":"assignments/submissions","error":str(e)})

(course_dir / "manifest_failed.json").write_text(json.dumps(failed, indent=2), encoding="utf-8")

print("\n✅ DONE")
print("Downloaded:", downloaded_count, "| Skipped (already existed):", skipped_count)
print("Failures logged:", len(failed))
print("See manifest_failed.json for details.")

In [ ]:
import os, re, json, time
from pathlib import Path
import requests
from tqdm import tqdm

base = os.environ["CANVAS_BASE_URL"].rstrip("/")
token = os.environ["CANVAS_TOKEN"].strip()

# ✅ Change this to your full archive folder
OUT_ROOT = Path("/content/drive/MyDrive/UNC_Canvas_Archive_All_Courses")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

session = requests.Session()
session.headers.update({"Authorization": f"Bearer {token}"})

def safe_name(x: str) -> str:
    x = re.sub(r"[^\w\-. ]+", "_", (x or "").strip())
    return (x[:140] if len(x) > 140 else x) or "untitled"

def get_all(url, params=None):
    params = params or {}
    params.setdefault("per_page", 100)
    out = []
    while url:
        r = session.get(url, params=params)
        r.raise_for_status()
        out.extend(r.json())
        link = r.headers.get("Link", "")
        nxt = None
        for part in link.split(","):
            if 'rel="next"' in part:
                nxt = part[part.find("<")+1:part.find(">")]
                break
        url = nxt
        params = None
    return out

def download(url: str, dest: Path):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0:
        return "skipped"
    with session.get(url, stream=True) as r:
        r.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
    return "downloaded"

print("Writing ALL courses to:", OUT_ROOT)

# ✅ This loops over ALL courses you already fetched in cell [7]
# (Make sure `courses` exists in memory; you printed "Total courses: 37")
for course_stub in courses:
    COURSE_ID = course_stub["id"]

    # Fetch full course detail (needed for syllabus_body)
    course = session.get(f"{base}/api/v1/courses/{COURSE_ID}").json()
    course_name = safe_name(course.get("name") or f"course_{COURSE_ID}")
    course_dir = OUT_ROOT / course_name

    # Resume protection (recommended). If you want to force re-run, delete the course folder.
    if (course_dir / "course.json").exists():
        print(f"Skipping already archived: {course_name}")
        continue

    course_dir.mkdir(parents=True, exist_ok=True)

    failed = []
    downloaded_count = 0
    skipped_count = 0

    print("\n==============================")
    print("Archiving:", course_name)
    print("Course ID:", COURSE_ID)
    print("==============================")

    (course_dir / "course.json").write_text(json.dumps(course, indent=2), encoding="utf-8")

    # 1) Files
    try:
        files = get_all(f"{base}/api/v1/courses/{COURSE_ID}/files")
        (course_dir / "files.json").write_text(json.dumps(files, indent=2), encoding="utf-8")

        files_dir = course_dir / "files"
        files_dir.mkdir(parents=True, exist_ok=True)

        for fobj in tqdm(files, desc=f"{course_name} files", unit="file"):
            name = safe_name(fobj.get("display_name") or fobj.get("filename") or f"file_{fobj['id']}")
            url = fobj.get("url")
            if not url:
                continue
            try:
                category = categorize_file(name)  # <-- your categorizer
                dest = files_dir / category / name
                status = download(url, dest)
                if status == "downloaded":
                    downloaded_count += 1
                else:
                    skipped_count += 1
            except Exception as e:
                failed.append({"type": "file", "name": name, "error": str(e)})
    except Exception as e:
        failed.append({"type": "files", "error": str(e)})

    # 2) Syllabus
    try:
        syllabus_html = course.get("syllabus_body") or ""
        (course_dir / "syllabus.html").write_text(syllabus_html, encoding="utf-8")
    except Exception as e:
        failed.append({"type": "syllabus", "error": str(e)})

    # 3) Modules + external links
    try:
        modules = get_all(f"{base}/api/v1/courses/{COURSE_ID}/modules")
        mod_dir = course_dir / "modules"
        mod_dir.mkdir(parents=True, exist_ok=True)

        external = set()
        for m in modules:
            mid = m["id"]
            items = get_all(f"{base}/api/v1/courses/{COURSE_ID}/modules/{mid}/items")
            (mod_dir / f"module_{mid}.json").write_text(
                json.dumps({"module": m, "items": items}, indent=2), encoding="utf-8"
            )
            for it in items:
                if it.get("external_url"):
                    external.add(it["external_url"])

        (course_dir / "external_links.json").write_text(json.dumps(sorted(external), indent=2), encoding="utf-8")
    except Exception as e:
        failed.append({"type": "modules", "error": str(e)})

    # 4) Pages
    try:
        pages = get_all(f"{base}/api/v1/courses/{COURSE_ID}/pages")
        pages_dir = course_dir / "pages"
        pages_dir.mkdir(parents=True, exist_ok=True)

        for p in pages:
            page = session.get(f"{base}/api/v1/courses/{COURSE_ID}/pages/{p['url']}").json()
            title = safe_name(page.get("title") or p["url"])
            (pages_dir / f"{title}.html").write_text(page.get("body") or "", encoding="utf-8")
    except Exception as e:
        failed.append({"type": "pages", "error": str(e)})

    # 5) Assignments + submissions (best effort)
    try:
        assignments = get_all(f"{base}/api/v1/courses/{COURSE_ID}/assignments")
        (course_dir / "assignments.json").write_text(json.dumps(assignments, indent=2), encoding="utf-8")

        subs_dir = course_dir / "submissions"
        subs_dir.mkdir(parents=True, exist_ok=True)

        for a in tqdm(assignments, desc=f"{course_name} submissions", unit="asg"):
            aid = a["id"]
            try:
                subs = get_all(
                    f"{base}/api/v1/courses/{COURSE_ID}/assignments/{aid}/submissions",
                    params={"include[]": ["attachments", "submission_history"]}
                )
                (subs_dir / f"assignment_{aid}_submissions.json").write_text(
                    json.dumps(subs, indent=2), encoding="utf-8"
                )

                for sub in subs:
                    for att in (sub.get("attachments") or []):
                        att_url = att.get("url")
                        att_name = safe_name(att.get("filename") or f"attachment_{att.get('id')}")
                        if att_url:
                            status = download(att_url, subs_dir / str(aid) / att_name)
                            if status == "downloaded":
                                downloaded_count += 1
                            else:
                                skipped_count += 1
            except Exception:
                continue
    except Exception as e:
        failed.append({"type": "assignments/submissions", "error": str(e)})

    (course_dir / "manifest_failed.json").write_text(json.dumps(failed, indent=2), encoding="utf-8")

    print("Downloaded:", downloaded_count, "| Skipped:", skipped_count, "| Failures:", len(failed))

    # Light throttle to be nice to Canvas
    time.sleep(0.5)

print("\n✅ ALL COURSES COMPLETE")

In [ ]:
from pathlib import Path
import json

OUT_ROOT = Path("/content/drive/MyDrive/UNC_Canvas_Archive_All_Courses")

expected_total = len(courses)
archived_courses = []
missing_courses = []
courses_with_failures = []

for course_stub in courses:
    cid = course_stub["id"]
    cname = safe_name(course_stub.get("name") or f"course_{cid}")
    cdir = OUT_ROOT / cname

    if (cdir / "course.json").exists():
        archived_courses.append((cid, cname))

        mf = cdir / "manifest_failed.json"
        if mf.exists():
            try:
                failures = json.loads(mf.read_text(encoding="utf-8"))
                if failures:
                    courses_with_failures.append((cid, cname, len(failures)))
            except Exception:
                pass
    else:
        missing_courses.append((cid, cname))

print("\n========== ARCHIVE DIFF SUMMARY ==========")
print("Courses available in Canvas:", expected_total)
print("Courses archived (folder + course.json):", len(archived_courses))
print("Courses missing:", len(missing_courses))
print("Courses with failures:", len(courses_with_failures))
print("==========================================\n")

if missing_courses:
    print("⚠️ Missing Courses:")
    for cid, cname in missing_courses:
        print(f" - {cid} | {cname}")

if courses_with_failures:
    print("\n⚠️ Courses With Failures:")
    for cid, cname, fcount in courses_with_failures:
        print(f" - {cid} | {cname} | Failures logged: {fcount}")
else:
    print("\n✅ No failure logs detected.")